In [1]:
# IMPORT LIBRARIES

import pandas as pd
import numpy as np



In [2]:
# DATA LOADING

file_path = r"C:\Users\alish\OneDrive\Desktop\customer_segmentation_anlaysis\customer_segmentation_analysis_5_years.xlsx"

excel_file = pd.ExcelFile(file_path)
print(excel_file.sheet_names)



['Transactions', 'Customers', 'Products', 'RFM_Analysis', 'Segment_Summary', 'Annual_Summary', 'Data_Dictionary']


In [3]:
# LOAD DATASETS

transactions_df = pd.read_excel(file_path, sheet_name="Transactions")
customer_df = pd.read_excel(file_path, sheet_name="Customers")
product_df = pd.read_excel(file_path, sheet_name="Products")



1. Transaction Data Exploration

In [ ]:
# TRANSACTIONS DATA EXPLORATION
#Basic Findings
print(transactions_df.shape)
print(transactions_df.columns)
transactions_df.info()
print(transactions_df.isnull().sum())
print(transactions_df["Transaction_ID"].duplicated().sum())


In [ ]:
# TRANSACTIONS DATA EXPLORATION
# Categorical Distributions

print(transactions_df["Purchase_Channel"].value_counts())
print(transactions_df["Payment_Method"].value_counts())
print(transactions_df["Transaction_Status"].value_counts())

In [ ]:
# TRANSACTIONS DATA EXPLORATION
# Numerical Statistics

print(transactions_df["Quantity"].describe())
print(transactions_df["Unit_Price_USD"].describe())
print(transactions_df["Gross_Spend_USD"].describe())
print(transactions_df["Discount_Pct"].describe())
print(transactions_df["Discount_Amount_USD"].describe())
print(transactions_df["Net_Spend_USD"].describe())
print(transactions_df["Recognized_Revenue_USD"].describe())
print(transactions_df["Cost_USD"].describe())
print(transactions_df["Profit_USD"].describe())

In [ ]:
# TRANSACTIONS DATA EXPLORATION
# Date Exploration

print(transactions_df["Transaction_Date"].min())
print(transactions_df["Transaction_Date"].max())
print(transactions_df["Transaction_Date"].dt.year.value_counts().sort_index())
print(transactions_df["Transaction_Date"].dt.month.value_counts().sort_index())

2. Customer Data Exploration

In [ ]:
# CUSTOMERS DATA EXPLORATION
  #Basic Findings
print(customer_df.shape)
print(customer_df.columns)
customer_df.info()
print(customer_df.isnull().sum())
print(customer_df.duplicated().sum())
print(customer_df["Customer_ID"].duplicated().sum())


In [ ]:
# CUSTOMERS DATA EXPLORATION
  #Categorical Distributions
print(customer_df["Gender"].value_counts())
print(customer_df["Region"].value_counts())
print(customer_df["City"].value_counts())
print(customer_df["Customer_Type"].value_counts())
print(customer_df["Acquisition_Channel"].value_counts())
print(customer_df["Age"].value_counts().sort_index())
print(customer_df["Signup_Year"].value_counts().sort_index())
print(customer_df["Signup_Date"].min())
print(customer_df["Signup_Date"].max())

In [ ]:
# CUSTOMERS DATA EXPLORATION
  #Numerical findings
print(customer_df["Age"].describe())
print(customer_df["Signup_Year"].describe())

3. Product Data Exploration

In [ ]:
# PRODUCTS DATA EXPLORATION
# Basic Findings
print(product_df.head())
print(product_df.shape)
print(product_df.columns)
print(product_df.info())
print(product_df.isnull().sum())
print(product_df.duplicated().sum())
print(product_df["Product_ID"].duplicated().sum())


In [ ]:
# PRODUCTS DATA EXPLORATION
# Categorical Distribution
print(product_df["Product_Name"].value_counts())
print(product_df["Category"].value_counts())


In [ ]:
# PRODUCTS DATA EXPLORATION
# Numerical Statistics
print(product_df["Unit_Price_USD"].describe())
print(product_df["Unit_Cost_USD"].describe())


In [ ]:
# PRODUCTS DATA EXPLORATION
# Calculations
#Calculate Unit Margin
product_df["Unit_Margin"] = (
    product_df["Unit_Price_USD"] -
    product_df["Unit_Cost_USD"]
)

display(product_df)

#Calculate Unit Margin%
product_df["Unit_Margin_Pct"] = (
    product_df["Unit_Margin"] /
    product_df["Unit_Price_USD"]
) * 100

display(product_df)


Check relationship

In [ ]:
#Check whether customer_id present in transactions_df is also present in customers_df
unmatched_customer_id=transactions_df.loc[
    ~transactions_df["Customer_ID"].isin(customer_df["Customer_ID"])
    ]
print("Unmatched_customer ids ", unmatched_customer_id.nunique())
print("Total unmatched customer ids ", len(unmatched_customer_id))

In [ ]:
#Check whether product_id present in transactions_df is also present in products_df
unmatched_product_id=transactions_df.loc[
    ~transactions_df["Product_ID"].isin(product_df["Product_ID"])
]
print("Unmatched product id is: ", unmatched_product_id.nunique())
print("No of unmatched product id is", unmatched_product_id)

Data Cleaning

In [ ]:
#Making copies of original data to perform cleaning activities
transaction_clean=transactions_df.copy()
products_clean=product_df.copy()
customers_clean=customer_df.copy()
#checking if it is copied
print(transactions_df.equals(transaction_clean))
print(product_df.equals(products_clean))
print(customer_df.equals(customers_clean))

In [ ]:
#Transaction Status Investigation
#Understand how each transaction status afftect profit and revenue generated
print(transaction_clean.groupby("Transaction_Status")[['Recognized_Revenue_USD', 'Profit_USD']].sum())

In [ ]:
#Cross verification if each returned and cancelled order generate revenue or profit
not_completed=transaction_clean[transaction_clean["Transaction_Status"]!= "Completed"]
#Counting how many transaction status except completed in each group generated non zero value for generating profit and revenue
not_completed.groupby("Transaction_Status").agg(
    Total_Records=("Transaction_ID","count"),
    Non_Zero_Revenue=("Recognized_Revenue_USD",lambda x:(x!=0).sum()),
    Non_Zero_Profit=("Profit_USD", lambda x:(x!=0).sum()))


In [ ]:
#Checking if quantity has 0 or negative values
invalid_quantity=transaction_clean[transaction_clean["Quantity"]<=0]
No_of_invalid_quantity=len(invalid_quantity)
print(invalid_quantity)
print(No_of_invalid_quantity)

In [ ]:
#checking if financial value is invalid: 0 or negative for 'Unit_Price_USD', 'Gross_Spend_USD', 'Discount_Pct', 'Discount_Amount_USD', 'Net_Spend_USD', 'Recognized_Revenue_USD', 'Cost_USD', 'Profit_USD'
# 1.unit_price
invalid_unit_price=transaction_clean[transaction_clean["Unit_Price_USD"]<=0]
no_of_invalid_unit_price=len(invalid_unit_price)
print(invalid_unit_price)
print(no_of_invalid_unit_price)


In [ ]:
#2.gross_spend
invalid_gross_spend=transaction_clean[transaction_clean["Gross_Spend_USD"]<=0]
no_of_invalid_gross_spend=len(invalid_gross_spend)
print(invalid_gross_spend)
print(no_of_invalid_gross_spend)


In [ ]:
#3.discount_amount
invalid_discount_amount=transaction_clean[transaction_clean["Discount_Amount_USD"]<0]
no_of_invalid_discount_amount=len(invalid_discount_amount)
print(invalid_discount_amount)
print(no_of_invalid_discount_amount)



In [ ]:
#4.Net_spend_usd
invalid_net_spend=transaction_clean[transaction_clean["Net_Spend_USD"]<=0]
no_of_invalid_net_spend=len(invalid_net_spend)
print(invalid_net_spend)
print(no_of_invalid_net_spend)


In [ ]:
#5.Recognized_Revenue_USD'
invalid_Recognized_Revenue=transaction_clean[
    (transaction_clean["Recognized_Revenue_USD"]<=0) &
    ( transaction_clean["Transaction_Status"]=="Completed")
    ]
no_of_invalid_Recognized_Revenue=len(invalid_Recognized_Revenue)
print(invalid_Recognized_Revenue)
print(no_of_invalid_Recognized_Revenue)


Empty DataFrame
Columns: [Transaction_ID, Transaction_Date, Customer_ID, Product_ID, Quantity, Unit_Price_USD, Gross_Spend_USD, Discount_Pct, Discount_Amount_USD, Net_Spend_USD, Recognized_Revenue_USD, Cost_USD, Profit_USD, Purchase_Channel, Payment_Method, Transaction_Status]
Index: []
0


In [ ]:
#6.Cost_USD
invalid_cost_usd=transaction_clean[
    (transaction_clean["Cost_USD"]<=0)&
    (transaction_clean["Transaction_Status"]=="Completed")]
no_of_invalid_cost_usd=len(invalid_cost_usd)
print(invalid_cost_usd)
print(no_of_invalid_cost_usd)


In [ ]:
#Profit_USD
invalid_profit_usd=transaction_clean[
    (transaction_clean["Profit_USD"]<0)&
    (transaction_clean["Transaction_Status"]=="Completed")]
no_of_invalid_profit_usd=len(invalid_profit_usd)
print(invalid_profit_usd)
print(no_of_invalid_profit_usd)


**Check Calculations:**

In [ ]:
#Checking Calculations
calculated_gross_spend=transaction_clean["Quantity"]*transaction_clean["Unit_Price_USD"]
gross_spend_difference=(calculated_gross_spend-transaction_clean["Gross_Spend_USD"])
print("No_OF_Wrong_Calculations", (gross_spend_difference.abs()>0.0000).sum())
print("Maximum_difference", gross_spend_difference.abs().max())

**Validation Result:**

The calculated Gross Spend was compared with the existing
Gross_Spend_USD column.

Although 1,549 records showed tiny floating-point differences,
the maximum difference was approximately 1.82 × 10⁻¹².

These differences are negligible and result from floating-point
precision. Therefore, the Gross Spend calculation is considered valid.
No correction is required.

In [50]:
#Check Calculations
# Calculate Net Spend independently

calculated_net_spend = (
    transaction_clean["Gross_Spend_USD"] -
    transaction_clean["Discount_Amount_USD"]
)

# Calculate differences

net_spend_difference = (
    calculated_net_spend -
    transaction_clean["Net_Spend_USD"]
)

# Count meaningful mismatches

print(
    "Number of wrong calculations:",
    (net_spend_difference.abs() > 0.000001).sum()
)

# Maximum difference

print(
    "Maximum difference:",
    net_spend_difference.abs().max()
)

Number of wrong calculations: 0
Maximum difference: 1.8189894035458565e-12


In [ ]:
#Check Calculations
# Calculate Discount Amount independently

calculated_discount_amount = (
    transaction_clean["Gross_Spend_USD"] *
    transaction_clean["Discount_Pct"]
)

# Calculate differences

discount_difference = (
    calculated_discount_amount -
    transaction_clean["Discount_Amount_USD"]
)

# Count meaningful mismatches

print(
    "Number of wrong calculations:",
    (discount_difference.abs() > 0.01).sum()
)

# Maximum difference

print(
    "Maximum difference:",
    discount_difference.abs().max()
)

In [ ]:
#Checking Calculations
# Calculate Profit independently

calculated_profit = (
    transaction_clean["Recognized_Revenue_USD"] -
    transaction_clean["Cost_USD"]
)

# Calculate differences

profit_difference = (
    calculated_profit -
    transaction_clean["Profit_USD"]
)

# Count meaningful mismatches

print(
    "Number of wrong calculations:",
    (profit_difference.abs() > 0.000001).sum()
)

# Maximum difference

print(
    "Maximum difference:",
    profit_difference.abs().max()
)

**Checking Inconsistent Categorical Columns**

In [ ]:
# Identify categorical columns in Transactions

categorical_columns = transaction_clean.select_dtypes(
    include="object"
).columns

# Display unique values in each categorical column

for column in categorical_columns:
    print(f"\nColumn: {column}")
    print(transaction_clean[column].unique())

In [ ]:
#Identify categorical columns in Customers
customer_categorical_columns=customers_clean.select_dtypes(
    include="object"
).columns
for c in customer_categorical_columns:
    print(f"Columns: \n {c}")
    print(customers_clean[c].unique())

In [ ]:
#Identify categorical columns in Products
products_categorical_columns=products_clean.select_dtypes(
    include="object"
).columns
for c in products_categorical_columns:
    print(f"Columns:\n {c}")
    print(products_clean[c].unique())


**IDENTIFYING OUTLIERS**

In [ ]:
#identifying outliers
#quantity
q1=transaction_clean["Quantity"].quantile(0.25)
q3=transaction_clean["Quantity"].quantile(0.75)
iqr=q3-q1
lower=q1-(1.5*iqr)
upper=q3+(1.5*iqr)
outlier=transaction_clean[
    (transaction_clean["Quantity"]<lower)|(transaction_clean["Quantity"]>upper)
]
no_of_outlier=len(outlier)
print("Q1 is: ",q1)
print("IQR is: ",iqr)
print("Q3 is: ",q3)
print("LOWER is: ",lower)
print("UPPER is: ",upper)
print("NO_OF_OUTLIER is: ",no_of_outlier)
outlier["Quantity"].value_counts().sort_index()

In [ ]:
#identifying outliers
#Unit_Price_USD
q1=transaction_clean["Unit_Price_USD"].quantile(0.25)
q3=transaction_clean["Unit_Price_USD"].quantile(0.75)
iqr=q3-q1
lower=q1-(1.5*iqr)
upper=q3+(1.5*iqr)
unit_price_outlier=transaction_clean[
    (transaction_clean["Unit_Price_USD"]<lower)|(transaction_clean["Unit_Price_USD"]>upper)
]
no_of_outlier=len(unit_price_outlier)
print("Q1 is: ",q1)
print("IQR is: ",iqr)
print("Q3 is: ",q3)
print("LOWER is: ",lower)
print("UPPER is: ",upper)
print("NO_OF_OUTLIER is: ",no_of_outlier)
unit_price_outlier["Unit_Price_USD"].value_counts().sort_index()


In [ ]:
#identifying outliers
#gross_spend
q1=transaction_clean["Gross_Spend_USD"].quantile(0.25)
q3=transaction_clean["Gross_Spend_USD"].quantile(0.75)
iqr=q3-q1
lower=q1-(1.5*iqr)
upper=q3+(1.5*iqr)
gross_spend_outlier=transaction_clean[
    (transaction_clean["Gross_Spend_USD"]<lower)|(transaction_clean["Gross_Spend_USD"]>upper)
]
no_of_outlier=len(gross_spend_outlier)
print("Q1 is: ",q1)
print("IQR is: ",iqr)
print("Q3 is: ",q3)
print("LOWER is: ",lower)
print("UPPER is: ",upper)
print("NO_OF_OUTLIER is: ",no_of_outlier)
gross_spend_outlier["Gross_Spend_USD"].value_counts().sort_index()

In [ ]:
#identifying outliers
#net_spend_usd
q1=transaction_clean["Net_Spend_USD"].quantile(0.25)
q3=transaction_clean["Net_Spend_USD"].quantile(0.75)
iqr=q3-q1
lower=q1-(1.5*iqr)
upper=q3+(1.5*iqr)
Net_Spend_USD_outlier=transaction_clean[
    (transaction_clean["Net_Spend_USD"]<lower)|(transaction_clean["Net_Spend_USD"]>upper)
]
no_of_Net_Spend_USD_outlier=len(Net_Spend_USD_outlier)
print("Q1 is: ",q1)
print("IQR is: ",iqr)
print("Q3 is: ",q3)
print("LOWER is: ",lower)
print("UPPER is: ",upper)
print("NO_OF_OUTLIER is: ",no_of_Net_Spend_USD_outlier)
Net_Spend_USD_outlier["Net_Spend_USD"].value_counts().sort_index()

Net_Spend_USD_outlier.nlargest(
    10, "Net_Spend_USD"
)[
    [
        "Product_ID",
        "Quantity",
        "Unit_Price_USD",
        "Gross_Spend_USD",
        "Discount_Pct",
        "Net_Spend_USD"
    ]
]

In [ ]:
#identifying outliers
#Recognized_Revenue_USD
q1=transaction_clean["Recognized_Revenue_USD"].quantile(0.25)
q3=transaction_clean["Recognized_Revenue_USD"].quantile(0.75)
iqr=q3-q1
lower=q1-(1.5*iqr)
upper=q3+(1.5*iqr)
Recognized_Revenue_USD_outlier=transaction_clean[
    (transaction_clean["Recognized_Revenue_USD"]<lower)|(transaction_clean["Recognized_Revenue_USD"]>upper)
]
no_of_outlier=len(Recognized_Revenue_USD_outlier)
print("Q1 is: ",q1)
print("IQR is: ",iqr)
print("Q3 is: ",q3)
print("LOWER is: ",lower)
print("UPPER is: ",upper)
print("NO_OF_OUTLIER is: ",no_of_outlier)
Recognized_Revenue_USD_outlier["Recognized_Revenue_USD"].value_counts().sort_index()

revenue_columns = [
    "Product_ID",
    "Quantity",
    "Unit_Price_USD",
    "Gross_Spend_USD",
    "Discount_Pct",
    "Net_Spend_USD",
    "Recognized_Revenue_USD",
    "Profit_USD",
    "Transaction_Status"
]

revenue_outlier_top10 = Recognized_Revenue_USD_outlier.sort_values(
    by="Recognized_Revenue_USD",
    ascending=False
)

revenue_outlier_top10[revenue_columns].head(10)

In [6]:
#identifying outliers
#Profit_USD
q1=transaction_clean["Profit_USD"].quantile(0.25)
q3=transaction_clean["Profit_USD"].quantile(0.75)
iqr=q3-q1
lower=q1-(1.5*iqr)
upper=q3+(1.5*iqr)
Profit_USD_outlier=transaction_clean[
    (transaction_clean["Profit_USD"]<lower)|(transaction_clean["Profit_USD"]>upper)
]
no_of_outlier=len(Profit_USD_outlier)
print("Q1 is: ",q1)
print("IQR is: ",iqr)
print("Q3 is: ",q3)
print("LOWER is: ",lower)
print("UPPER is: ",upper)
print("NO_OF_OUTLIER is: ",no_of_outlier)
Profit_USD_outlier["Profit_USD"].value_counts().sort_index()

revenue_columns = [
    "Product_ID",
    "Quantity",
    "Unit_Price_USD",
    "Gross_Spend_USD",
    "Discount_Pct",
    "Net_Spend_USD",
    "Recognized_Revenue_USD",
    "Profit_USD",
    "Transaction_Status"
]

revenue_outlier_top10 = Recognized_Revenue_USD_outlier.sort_values(
    by="Profit_USD",
    ascending=False
)

revenue_outlier_top10[revenue_columns].head(10)

NameError: name 'transaction_clean' is not defined

**RFM Preparation**

In [ ]:
rfm_transactions=transaction_clean[transaction_clean["Transaction_Status"]=="Completed"].copy()
#keeping only required columns
rfm_transactions=rfm_transactions[
    [
        "Transaction_ID", "Customer_ID", "Transaction_Date","Recognized_Revenue_USD"
        
        
    ]
]
rfm_transactions.head()

In [ ]:
# Find the latest transaction date

latest_transaction_date = rfm_transactions["Transaction_Date"].max()


# Set reference date to one day after the latest transaction

rfm_reference_date = (
    latest_transaction_date + pd.Timedelta(days=1)
)

print("Latest Transaction Date:", latest_transaction_date)
print("RFM Reference Date:", rfm_reference_date)

In [ ]:
# Find the latest transaction date

latest_transaction_date = rfm_transactions["Transaction_Date"].max()


# Set reference date to one day after the latest transaction

rfm_reference_date = (
    latest_transaction_date + pd.Timedelta(days=1)
)

print("Latest Transaction Date:", latest_transaction_date)
print("RFM Reference Date:", rfm_reference_date)

In [ ]:
print("Total completed transaction records:", len(rfm_transactions))

print(
    "Unique customers:",
    rfm_transactions["Customer_ID"].nunique()
)

print(
    "Unique transactions:",
    rfm_transactions["Transaction_ID"].nunique()
)

print(
    "Total recognized revenue:",
    rfm_transactions["Recognized_Revenue_USD"].sum()
)

print(
    "Missing values:\n",
    rfm_transactions.isnull().sum()
)

In [ ]:
rfm = rfm_transactions.groupby("Customer_ID").agg(
    Recency=("Transaction_Date", lambda x: (
        rfm_reference_date - x.max()
    ).days),

    Frequency=("Transaction_ID", "nunique"),

    Monetary=("Recognized_Revenue_USD", "sum")
)

display(rfm.head())
print(rfm.shape)
print(rfm.describe())

**RFM SCORING**

In [ ]:
# Calculate RFM scores

rfm["R_Score"] = pd.qcut(
    rfm["Recency"],
    q=5,
    labels=[5, 4, 3, 2, 1]
)

rfm["F_Score"] = pd.qcut(
    rfm["Frequency"].rank(method="first"),
    q=5,
    labels=[1, 2, 3, 4, 5]
)

rfm["M_Score"] = pd.qcut(
    rfm["Monetary"],
    q=5,
    labels=[1, 2, 3, 4, 5]
)

display(rfm.head())

In [ ]:
rfm["RFM_Score"] = (
    rfm["R_Score"].astype(str) +
    rfm["F_Score"].astype(str) +
    rfm["M_Score"].astype(str)
)

display(rfm.head())